# Generating Exercise Notebooks

Suppose you want to learn backpropagation — really learn it, not just recognize it on a multiple-choice exam. The fastest path is to sit down and implement it: struggle with the shapes, debug the chain rule, verify numerically. That friction *is* the learning. Now consider what happens when an LLM is one tab away. The temptation is structural, not malicious: why spend three hours on a derivation when a single prompt returns working code in thirty seconds?

The same technology has a sharper edge. Rather than generating answers, we can use LLMs to generate [challenges]{.mark}. Take a polished explanatory notebook — one that already walks through every derivation and implementation — and produce an **exercise version** in which the reader must fill in the code themselves. The explanatory prose and structure remain; the implementations become `TODO` scaffolds requiring active reconstruction. This notebook builds exactly that pipeline.

## Notebook Anatomy

A `.ipynb` file is a JSON document. At the top level it carries `nbformat`, `nbformat_minor`, `metadata`, and a list of `cells`. Each cell has a `cell_type` (one of `"markdown"`, `"code"`, or `"raw"`), a short `id` string, a `metadata` dict, and a `source` field containing the cell's text — either a single string or a list of strings (one element per line), depending on the tool that last saved the file. Code cells additionally carry `execution_count` and `outputs`.

The `source` field is what matters for our purposes. For a markdown cell it holds prose or LaTeX; for a code cell it holds executable Python. The rest — outputs, execution counts, kernel metadata — we can largely ignore or reset when generating the exercise notebook.

In [ ]:
import json
import uuid
from pathlib import Path

from openai import OpenAI


We load a notebook and print a structural summary to get our bearings:

In [ ]:
def load_notebook(path: str | Path) -> dict:
    """Load a Jupyter notebook from disk and return its parsed JSON."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_source(cell: dict) -> str:
    """Return the cell source as a single string regardless of storage format."""
    src = cell["source"]
    if isinstance(src, list):
        return "".join(src)
    return src


def summarize_notebook(nb: dict) -> None:
    """Print a brief structural summary of a notebook."""
    cells = nb["cells"]
    code_cells = [c for c in cells if c["cell_type"] == "code"]
    md_cells   = [c for c in cells if c["cell_type"] == "markdown"]
    print(f"Total cells  : {len(cells)}")
    print(f"  Markdown   : {len(md_cells)}")
    print(f"  Code       : {len(code_cells)}")
    print(f"nbformat     : {nb['nbformat']}.{nb['nbformat_minor']}")


nb = load_notebook("03-svd.ipynb")
summarize_notebook(nb)


**Cell role taxonomy.** Not every code cell is a candidate for conversion. We distinguish four roles:

- `setup` — imports, configuration, constants, random seeds. No learning signal; the reader should not be asked to reconstruct these.
- `implementation` — defines a class, function, or non-trivial algorithm. This is where understanding lives, and therefore where exercises belong.
- `visualization` — produces a plot or formatted table. Pedagogically secondary; scaffolding these creates noise, not learning.
- `utility` — short expressions: calling a helper, printing a result, running a timing check. Usually not worth exercising.

The goal of classification is to isolate `implementation` cells so we can transform only those.

## Cell Classification

We ask an LLM to classify each code cell. The prompt gives the model the cell source and a closed set of four labels, and asks for exactly one label plus a one-sentence rationale. Using `gpt-4o-mini` keeps cost negligible — classification does not require a large model.

In [ ]:
CLASSIFY_PROMPT = """\
You are classifying a Jupyter notebook code cell into exactly one of four roles.

Roles:
  setup          - imports, configuration, constants, seeds, installs
  implementation - defines a class, function, or non-trivial algorithm
  visualization  - produces a plot, chart, or formatted display
  utility        - short expressions, print calls, assertions, timing

Return your answer as a JSON object with two keys:
  "role"   : one of the four role strings above
  "reason" : one sentence explaining your choice

Do not include any other text. The cell source follows.

---
{source}
---
"""


def classify_cell(client: OpenAI, source: str) -> dict:
    """Classify a single code cell. Returns {role, reason}."""
    prompt = CLASSIFY_PROMPT.format(source=source)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"role": "utility", "reason": "parse error — defaulted to utility"}


def classify_notebook_cells(nb: dict, client: OpenAI) -> list[dict]:
    """Return a classification record for every code cell in the notebook.

    Each record is a dict with keys:
      index  : original cell index in nb['cells']
      source : normalized cell source string
      role   : classified role string
      reason : one-sentence rationale from the LLM
    """
    results = []
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue
        source = normalize_source(cell)
        if not source.strip():
            results.append({"index": i, "source": source, "role": "utility", "reason": "empty cell"})
            continue
        classification = classify_cell(client, source)
        results.append({
            "index": i,
            "source": source,
            "role": classification.get("role", "utility"),
            "reason": classification.get("reason", ""),
        })
    return results


We run classification on the SVD notebook and print the results:

In [ ]:
#| output: false
client = OpenAI()
classifications = classify_notebook_cells(nb, client)


In [ ]:
for item in classifications:
    preview = item["source"].split("\n")[0][:60]
    print(f"[{item['role']:>14}]  {preview}")
    print(f"               {item['reason']}")
    print()


## Exercise Cell Generation

The core design constraint is: [preserve enough structure to guide, remove enough to require thinking]{.mark}. Concretely, we keep function and class signatures, docstrings, type hints, and import statements. We replace all statement bodies with `TODO` stubs — short plain-English instructions describing what to implement without revealing how.

A `difficulty` parameter controls how much additional context survives. At `easy`, helper function names and variable names are left in place as hints. At `hard`, only the signature and docstring remain. `medium` is the default: the structure is clear but the implementation path is not.

In [ ]:
EXERCISE_PROMPT = """\
You are converting a Python code cell from an explanatory notebook into an exercise cell.

Rules:
  1. Keep all import statements exactly as-is.
  2. Keep all function and class signatures exactly as-is (name, parameters, return type).
  3. Keep all docstrings exactly as-is.
  4. Keep all module-level type aliases and constants exactly as-is.
  5. Replace every function or method body with:
       # TODO: implement
       raise NotImplementedError
  6. Replace any non-trivial multi-line block (loop body, if body, comprehension body)
     with a single comment:
       # TODO: <short plain-English description of what the block should do>
  7. Do NOT add any explanation or commentary outside the code.
  8. Return only the transformed Python source, with no markdown fences.

Difficulty level: {difficulty}
  easy   - leave variable names and helper call names in place as hints
  medium - remove helper call names; keep structural shape
  hard   - remove all hints; only signature and docstring remain

Original cell:
---
{source}
---
"""


def generate_exercise_cell(
    client: OpenAI,
    source: str,
    difficulty: str = "medium",
) -> str:
    """Transform an implementation cell into a TODO-scaffolded exercise."""
    prompt = EXERCISE_PROMPT.format(source=source, difficulty=difficulty)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


We pick the first `implementation` cell from the SVD notebook and compare the original against the generated exercise:

In [ ]:
impl_items = [item for item in classifications if item["role"] == "implementation"]
example = impl_items[0]

print("=== ORIGINAL ===")
print(example["source"])
print()
exercised = generate_exercise_cell(client, example["source"], difficulty="medium")
print("=== EXERCISE ===")
print(exercised)


**Hint generation.** For readers who want guided practice rather than a cold-start recall test, we optionally generate a pseudocode hint. The hint contains just enough to jog memory — no Python syntax, no helper names — and is wrapped in a Quarto collapsible callout so it is hidden by default:

In [ ]:
HINT_PROMPT = """\
Write a brief pseudocode hint for implementing the following Python function.
The hint should be 3-6 lines of plain pseudocode (not Python syntax).
Do not reveal variable names, helper function names, or any implementation
detail beyond the high-level algorithmic steps.
Return only the pseudocode, no other text.

Function:
---
{source}
---
"""


def generate_hint(client: OpenAI, source: str) -> str:
    """Generate a collapsible pseudocode hint callout for an implementation cell."""
    prompt = HINT_PROMPT.format(source=source)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    pseudocode = response.choices[0].message.content.strip()
    return (
        ':::{.callout-tip collapse="true"}\n'
        "## Hint\n"
        f"{pseudocode}\n"
        ":::"
    )


## End-to-End Pipeline

We assemble the full pipeline into `generate_exercises`. It (1) loads the source notebook, (2) classifies all code cells, (3) transforms each `implementation` cell into a TODO-scaffolded exercise, (4) optionally injects hint callouts as markdown cells immediately following each exercise cell, and (5) writes the result to disk.

In [ ]:
def new_markdown_cell(source: str) -> dict:
    """Return a minimal Jupyter markdown cell dict."""
    return {
        "cell_type": "markdown",
        "id": uuid.uuid4().hex[:8],
        "metadata": {},
        "source": source,
    }


def new_code_cell(source: str) -> dict:
    """Return a minimal Jupyter code cell dict."""
    return {
        "cell_type": "code",
        "execution_count": None,
        "id": uuid.uuid4().hex[:8],
        "metadata": {},
        "outputs": [],
        "source": source,
    }


def generate_exercises(
    source_path: str | Path,
    output_path: str | Path,
    client: OpenAI,
    difficulty: str = "medium",
    include_hints: bool = False,
) -> dict:
    """Convert an explanatory notebook into an exercise notebook.

    Parameters
    ----------
    source_path:
        Path to the source `.ipynb` file.
    output_path:
        Path where the exercise notebook will be written.
    client:
        An initialized OpenAI client.
    difficulty:
        One of ``"easy"``, ``"medium"``, or ``"hard"``. Controls how much
        scaffolding is preserved in exercise cells.
    include_hints:
        If True, insert a collapsible pseudocode hint callout after each
        exercise cell.

    Returns
    -------
    dict
        Summary statistics: total cells, implementation cells converted,
        hints injected.
    """
    nb = load_notebook(source_path)
    classifications = classify_notebook_cells(nb, client)      # <1>
    impl_index = {                                              # <2>
        item["index"]: item["source"]
        for item in classifications
        if item["role"] == "implementation"
    }

    new_cells = []
    hints_added = 0

    for i, cell in enumerate(nb["cells"]):                     # <3>
        src = normalize_source(cell)
        is_code = cell["cell_type"] == "code"

        if not is_code or i not in impl_index:
            rebuilt = {**cell, "source": src}
            if is_code:
                rebuilt["outputs"] = []
                rebuilt["execution_count"] = None
            new_cells.append(rebuilt)
            continue

        exercise_src = generate_exercise_cell(                  # <4>
            client, src, difficulty=difficulty
        )
        new_cells.append(new_code_cell(exercise_src))

        if include_hints:                                       # <5>
            hint_src = generate_hint(client, src)
            new_cells.append(new_markdown_cell(hint_src))
            hints_added += 1

    out_nb = {
        "cells": new_cells,
        "metadata": nb["metadata"],
        "nbformat": nb["nbformat"],
        "nbformat_minor": nb["nbformat_minor"],
    }
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out_nb, f, indent=1, ensure_ascii=False)

    return {
        "total_cells": len(nb["cells"]),
        "impl_cells_converted": len(impl_index),
        "hints_injected": hints_added,
        "output_path": str(output_path),
    }


1. All code cells are classified upfront. We build `impl_index` — a mapping from cell position to source — before the transformation loop so we do not re-call the API during iteration.
2. `impl_index` is keyed by the original cell index. Non-implementation cells (setup, visualization, utility, markdown) pass through verbatim with outputs cleared.
3. We iterate over the original cell list rather than the classification results so that markdown cells — which are never classified — pass through unchanged.
4. Each implementation cell is replaced with a TODO-scaffolded version. A fresh `id` is generated for the new cell.
5. Hints are inserted as standalone markdown cells immediately after their corresponding exercise cell, so the reader can expand them inline without leaving the notebook.

Running the full pipeline on the SVD notebook:

In [ ]:
#| output: false
stats = generate_exercises(
    source_path="03-svd.ipynb",
    output_path="03-svd-exercises.ipynb",
    client=client,
    difficulty="medium",
    include_hints=True,
)


In [ ]:
for key, val in stats.items():
    print(f"{key:<25} {val}")

# Spot-check: confirm TODO appears in the generated notebook
ex_nb = load_notebook(stats["output_path"])
todo_count = sum(
    1 for c in ex_nb["cells"]
    if c["cell_type"] == "code" and "TODO" in normalize_source(c)
)
print(f"\nCells with TODO stubs: {todo_count}")


## Quality and Design Considerations

**What makes a good exercise?** Three conditions matter. First, *clear intent*: the reader must know what they are supposed to implement. The docstring and the surrounding markdown cell — both of which we preserve — carry this load. An undocumented function with a hollow body is a puzzle, not an exercise. Second, *right level of scaffolding*: the `difficulty` parameter controls how much is left in. `easy` leaves helper names visible; `hard` strips them. The correct level depends on the audience and how recently the concept was introduced. Third, *testable output*: the exercise notebook should include the same assertions and verification expressions as the original. We do not transform `utility` cells, so these survive intact.

**Failure modes.** Two are common. *Over-scaffolding*: the LLM is cautious and leaves so many hints in the TODO comments that the reader can reconstruct the implementation by reading rather than thinking. This is almost as bad as giving the answer directly. The `hard` difficulty mode partially mitigates this, but human review of the generated cells is the reliable fix. *Under-scaffolding*: at `hard` difficulty on a complex cell, the exercise may be so sparse that a reader without prerequisite context cannot proceed — especially when the cell depends on definitions established earlier in the notebook. Keeping docstrings precise is the most effective countermeasure.

**Difficulty calibration.** A notebook benefits from *mixed* difficulty: early cells at `easy` to build orientation, later cells at `hard` to force synthesis. The pipeline can accept a per-cell difficulty map keyed by cell index. A second-pass solvability check — asking the LLM whether each exercise is likely solvable without additional context — is a useful quality gate:

In [ ]:
SOLVABLE_PROMPT = """\
Given the exercise cell below, assess whether a reader who has read the preceding
explanatory notebook (but not this cell's original implementation) could solve it
without additional hints. Answer with a JSON object:
  "solvable" : true or false
  "reason"   : one sentence

Exercise cell:
---
{source}
---
"""


def assess_solvability(client: OpenAI, exercise_source: str) -> dict:
    """Return {solvable: bool, reason: str} for an exercise cell."""
    prompt = SOLVABLE_PROMPT.format(source=exercise_source)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"solvable": True, "reason": "parse error — assumed solvable"}


:::{.callout-caution}
The pipeline sends the full source of every implementation cell to the API. Do not use it on notebooks containing reference solutions to graded assignments — the content will leave your environment. For sensitive material, route through a locally hosted model.
:::

## CLI Wrapper

We wrap the pipeline in a small `argparse` CLI so it can be invoked without opening a notebook:

```{.python filename="exercise_gen.py"}
import argparse
import json
from pathlib import Path
from openai import OpenAI


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Convert an explanatory notebook into an exercise notebook."
    )
    parser.add_argument("source", type=Path, help="Source .ipynb file")
    parser.add_argument(
        "--output", "-o", type=Path, default=Path("exercises"),
        help="Output directory (default: exercises/)",
    )
    parser.add_argument(
        "--difficulty", "-d",
        choices=["easy", "medium", "hard"],
        default="medium",
    )
    parser.add_argument(
        "--hints", action="store_true",
        help="Inject collapsible pseudocode hint callouts",
    )
    args = parser.parse_args()

    output_file = args.output / args.source.name
    stats = generate_exercises(
        source_path=args.source,
        output_path=output_file,
        client=OpenAI(),
        difficulty=args.difficulty,
        include_hints=args.hints,
    )
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()
```

Example invocation:

```{.bash filename="$ (local)"}
python exercise_gen.py notebooks/tooling/03-svd.ipynb --output exercises/ --difficulty medium --hints
```

---

The most educationally valuable thing an LLM can do in a learning context is not answer questions — it is generate better questions. A tool that converts polished explanatory material into structured exercises forces active recall, requires genuine implementation, and scales to any notebook in a corpus. The gap between reading a derivation and writing one is where understanding actually forms; this pipeline exists to widen that gap deliberately.

&#9632;